In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import geopandas as gpd
import statsmodels.api as sm
import pylab as pl
import time
import matplotlib.dates as mdates
from datetime import datetime

In [ ]:
#载入中文字体
import matplotlib
matplotlib.rc("font",family='DengXian')

### 准备数据

In [ ]:
#读取数据
df=pd.read_csv(r"E:\000Data\03school\03Table\0723新数据识别\00重要！合并后的数据\活动信息5月_未筛选.csv",header=0)
df

In [ ]:
df.info()

In [ ]:
#外卖领取点信息
diliplace=pd.read_excel(r"E:\000Data\03school\02数据\外卖领取点信息.xlsx",header=0)
diliplace

时间转换

In [ ]:
def secondshift(x):
    return x.hour*3600+x.minute*60+x.second

df['Time']=pd.to_datetime(df['Time'])
df["Day"]=df["Time"].apply(lambda x: x.strftime("%Y/%m/%d"))
df["Second"]=df["Time"].apply(secondshift)
sametime=[]
for i in range(df.shape[0]-1):
    if abs(df.loc[i+1,"Second"]-df.loc[i,"Second"])<=15  :
        sametime.append(i)
df.drop(df.index[sametime],inplace=True)
df

In [ ]:
df.columns

重新排序及信息更新

In [ ]:
def Informationupdate(df, diliplace):
    # 按照 'Day'、'St_ID' 和 'Time' 对数据框 df 排序
    df = df.sort_values(by=['Day', 'St_ID', 'Time'])  
    # 按照 'Place' 进行左关联
    df = df.merge(diliplace, on='Place', how='left')
    return df

df = Informationupdate(df, diliplace)
df

### 筛选特定就餐时间内活动

In [ ]:
# 验证是否为合法的午餐时间
def validlunch(second):
    # 判断时间是否在午餐的时间范围内（10:30 - 13:30）
    if 37800 < second < 48600:
        return True  
    else:
        return False 

# 验证是否为合法的晚餐时间
def validdinner(second):
    # 判断时间是否在晚餐的时间范围内（16:30 - 19:30）
    if 59400 < second < 70200:
        return True  
    else:
        return False 

### 判断是否在校就餐

In [ ]:
# 判断是否满足指定类型的餐食
def mealisnot(x, y):
    try:
        if y == "早餐" or y == "午餐" or y == "晚餐":
            x = x.reset_index(drop=True)
            number = x.shape[0] 
            for i in range(number):
                if x.loc[i, "Type_s"] == y or x.loc[i, "Type_s"] == "其他":
                    return True 
                    break 
                else:
                    pass 
            return False 
    except:
        return "Error! Choose the right type of meal!"
    

In [ ]:
# 初始化
df['Lunch'] = df.apply(lambda row: 1 if validlunch(row['Second']) else 0, axis=1)
df['Dinner'] = df.apply(lambda row: 1 if validdinner(row['Second']) else 0, axis=1)
df['Lunchtype'] = 0
df['Dinnertype'] = 0

# 分组并同时进行所有检查和更新
grouped = df.groupby(['Day', 'St_ID'])
for name, group in grouped:
    # 初始化为0，如果有餐则更新为1
    lunch_sum = group['Lunch'].sum()
    dinner_sum = group['Dinner'].sum()

    # 标记午餐类型
    if lunch_sum == 0:
        df.loc[group.index, 'Lunchtype'] = 0#无法识别
    elif mealisnot(group, "午餐"):
        df.loc[group.index, 'Lunchtype'] = 2#在校就餐
    else:
        df.loc[group.index, 'Lunchtype'] = 1#还未识别的
    
    # 标记晚餐类型
    if dinner_sum == 0:
        df.loc[group.index, 'Dinnertype'] = 0
    elif mealisnot(group, "晚餐"):
        df.loc[group.index, 'Dinnertype'] = 2
    else:
        df.loc[group.index, 'Dinnertype'] = 1

# 输出结果查看
df

In [ ]:
df.to_csv(r'E:\000Data\03school\05BKS\外卖识别_1.csv',index=False)

In [ ]:
df=pd.read_csv(r"E:\000Data\03school\05BKS\外卖识别_1.csv",header=0)
df

In [ ]:
df['Time']=pd.to_datetime(df['Time'])


In [ ]:
# 将 'Day' 列转换回 datetime 类型，以便进行日期比较
df['Day'] = pd.to_datetime(df['Day'])

# 筛选 2023/05/15 到 2023/05/20 期间的行
start_date = pd.to_datetime('2023-05-15')
end_date = pd.to_datetime('2023-05-16')

# 使用布尔索引选取指定日期范围的行
filtered_df = df[(df['Day'] >= start_date) & (df['Day'] < end_date)]

# 更新 df 为筛选后的 DataFrame
df = filtered_df

# 输出筛选后的 DataFrame
df

### 判断进出次数类型

In [ ]:
def Gatetimes(df):
    df = df.sort_values(by='Time')
    df['Gatetimes'] = 0

    for i in range(len(df)):
        if pd.isna(df.iloc[i]['DeliveryPlace']):
            df.iloc[i, df.columns.get_loc('Gatetimes')] = 0
        else:
            forward_count = 0
            for j in range(i - 1, -1, -1):
                #如果'DeliveryPlace'的值与i行的不同了，中断计数
                if pd.isna(df.iloc[j]['DeliveryPlace']) or df.iloc[j]['DeliveryPlace'] != df.iloc[i]['DeliveryPlace']:
                    break
                forward_count += 1

            backward_count = 0
            for k in range(i + 1, len(df)):
                #如果'DeliveryPlace'的值与i行的不同了，中断计数
                if pd.isna(df.iloc[k]['DeliveryPlace']) or df.iloc[k]['DeliveryPlace'] != df.iloc[i]['DeliveryPlace']:
                    break
                backward_count += 1

            df.iloc[i, df.columns.get_loc('Gatetimes')] = forward_count + backward_count + 1

    return df

In [ ]:
def process_meal(df, meal_type_key, meal_key):
    meal_df = df[(df[meal_type_key] == 1) & (df[meal_key] == 1)]
    grouped = meal_df.groupby(['Day', 'St_ID'])

    for name, group in grouped:
        original_indices = group.index
        updated_group = Gatetimes(group.copy())
        df.loc[original_indices, ['Gatetimes']] = updated_group[['Gatetimes']]

    grouped2 = df[df[meal_type_key] == 1].groupby(['Day', 'St_ID'])
    for name, group in grouped2:
        if group[meal_key].eq(1).any():
            if (group['Gatetimes'] >= 2).any():
                df.loc[group.index, meal_type_key] = 5
            elif (group['Gatetimes'] == 1).any():
                df.loc[group.index, meal_type_key] = 6
            else:
                df.loc[group.index, meal_type_key] = 7



# 处理午餐
process_meal(df, 'Lunchtype', 'Lunch')

# 处理晚餐
process_meal(df, 'Dinnertype', 'Dinner')

# 输出结果查看
df

In [ ]:
df.to_csv(r'E:\000Data\03school\05BKS\外卖识别_2.csv',index=False)

### 计算时间间隔

In [ ]:
def Timetype_1(df):
    # 排序数据框，按照 'Time' 列升序排列
    df = df.sort_values(by='Time')

    df['OutTimes'] = 0
    df['DiliTimes'] = 0

    # 筛选 'Lunchtype' 为 1 且 'Gatetimes' 大于等于 2 的行
    df_filtered = df[(df['Gatetimes'] >= 2)]

    # 遍历筛选后的行，计算相邻行中 'DeliveryPlace' 相同的组，并计算时间差
    for i in range(len(df_filtered) - 1):
        if df_filtered.iloc[i]['DeliveryPlace'] == df_filtered.iloc[i+1]['DeliveryPlace']:
            time_diff =df_filtered.iloc[i+1]['Second'] - df_filtered.iloc[i]['Second']

            if time_diff > 900:
                df.loc[df_filtered.index[i], 'OutTimes'] += 1
            elif time_diff <= 900:
                df.loc[df_filtered.index[i], 'DiliTimes'] += 1

    return df

In [ ]:
def process_meal2(df, meal_type_key, meal_key):
    meal_df = df[(df[meal_type_key] == 5) & (df[meal_key] == 1)]
    grouped = meal_df.groupby(['Day', 'St_ID'])

    for name, group in grouped:
        original_indices = group.index
        updated_group = Timetype_1(group.copy())
        df.loc[original_indices, ['OutTimes']] = updated_group[['OutTimes']]
        df.loc[original_indices, ['DiliTimes']] = updated_group[['DiliTimes']]

    grouped2 = df[df[meal_type_key] == 5].groupby(['Day', 'St_ID'])
    for name, group in grouped2:
        if group[meal_key].eq(1).any():
            if (group['OutTimes'] == 1).any():
                df.loc[group.index, meal_type_key] = 3
            elif (group['DiliTimes'] == 1).any():
                df.loc[group.index, meal_type_key] = 4


# 处理午餐
process_meal2(df, 'Lunchtype', 'Lunch')

# 处理晚餐
process_meal2(df, 'Dinnertype', 'Dinner')

# 输出结果查看
df

In [ ]:
df.to_csv(r'E:\000Data\03school\05BKS\外卖识别_3.csv',index=False)

In [ ]:
# 筛选 'Lunchtype' 为 1 的行，并分组
grouped = df.groupby(['Day', 'St_ID'])

# 对每个分组保留唯一行，并只保留特定字段
unique_rows = grouped.agg({
    'Day': 'first',  # 保留'Day'
    'St_ID': 'first',  # 保留'St_ID'
    'OutTimes': 'sum',  # 计算总'OutTimes'
    'DiliTimes': 'sum',  # 计算总'DiliTimes'
    'Lunch': 'first',  # 保留'Lunch'
    'Lunchtype': 'first',  # 保留'Lunchtype'
    'Dinner': 'first',  # 保留'Dinner'
    'Dinnertype': 'first'  # 保留'Dinnertype'
})

# 分类计数和计算占比
def calculate_counts_and_proportions(df, column):
    counts = df[column].value_counts().reset_index()
    counts.columns = [column, 'Counts']
    counts['Proportion'] = (counts['Counts'] / counts['Counts'].sum()) * 100
    return counts

# 计算'Lunchtype'、'Dinnertype'、'OutTimes'和'DiliTimes'的分类计数及占比
lunchtype_stats = calculate_counts_and_proportions(unique_rows, 'Lunchtype')
dinnertype_stats = calculate_counts_and_proportions(unique_rows, 'Dinnertype')
outtimes_stats = calculate_counts_and_proportions(unique_rows, 'OutTimes')
dilitimes_stats = calculate_counts_and_proportions(unique_rows, 'DiliTimes')

# 输出结果
print("Lunchtype Statistics:\n", lunchtype_stats)
print("Dinnertype Statistics:\n", dinnertype_stats)
print("OutTimes Statistics:\n", outtimes_stats)
print("DiliTimes Statistics:\n", dilitimes_stats)